In [2]:
import requests
import pandas as pd
from pathlib import Path
import yaml
from datetime import datetime
from tqdm import tqdm

### **Get (Current) Index Constituents**

In [ ]:
CONFIG_PATH = Path("../data_apiKey/config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
API_KEY = config["keys"]["fmp_api_key"]

INDEX_ENDPOINTS = {
    "SPX": "https://financialmodelingprep.com/stable/sp500-constituent",
    "NASDAQ": "https://financialmodelingprep.com/stable/nasdaq-constituent",
    "DOWJONES": "https://financialmodelingprep.com/stable/dowjones-constituent"
}

OUTPUT_DIR = Path("../database/indices_constituents")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_constituents(url):
    r = requests.get(f"{url}?apikey={API_KEY}")
    r.raise_for_status()
    return r.json()


def get_profile(symbol):
    url = f"https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={API_KEY}"
    r = requests.get(url)
    if r.status_code != 200:
        return {}
    data = r.json()
    return data[0] if data else {}


def build_dataframe(constituents):
    rows = []
    for c in constituents:
        symbol = c["symbol"]
        profile = get_profile(symbol)
        rows.append({
            "symbol": symbol,
            "cik": profile.get("cik"),
            "isin": profile.get("isin"),
            "cusip": profile.get("cusip"),
            "sector": profile.get("sector"),
            "country": profile.get("country"),
            "currency": profile.get("currency"),
            "marketCap": profile.get("marketCap"),
            "beta": profile.get("beta"),
            "exchange": profile.get("exchange"),
            "fullTimeEmployees": profile.get("fullTimeEmployees"),
            "lastDividend": profile.get("lastDividend"),
            "price": profile.get("price"),
        })

    df = pd.DataFrame(rows).set_index("symbol")
    return df


def save_parquet(index_name, df):
    df = df.sort_values("marketCap", ascending=False)
    date_str = datetime.now().strftime("%y%m%d")
    filename = f"{index_name}_{date_str}_constituents.parquet"
    df.to_parquet(OUTPUT_DIR / filename)
    print(f"Saved: {filename} with shape {df.shape}")


def main():
    for name, url in INDEX_ENDPOINTS.items():
        print(f"Fetching {name} constituents...")
        constituents = get_constituents(url)
        df = build_dataframe(constituents)
        save_parquet(name, df)
    return df

df = main()


Fetching SPX constituents...
Saved: SPX_251128_constituents.parquet with shape (503, 12)
Fetching NASDAQ constituents...
Saved: NASDAQ_251128_constituents.parquet with shape (101, 12)
Fetching DOWJONES constituents...
Saved: DOWJONES_251128_constituents.parquet with shape (30, 12)


### **Get Financial Statements**

In [ ]:
CONFIG_PATH = Path("../data_apiKey/config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
API_KEY = config["keys"]["fmp_api_key"]

# 目标目录
INPUT_DIR = Path("../database/indices_constituents")
OUTPUT_DIR = Path("../database/financial_statements")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# API endpoints
FIN_STAT_ENDPOINTS = {
    "income_statement": "https://financialmodelingprep.com/stable/income-statement",
    "balance_sheet": "https://financialmodelingprep.com/stable/balance-sheet-statement",
    "cash_flow": "https://financialmodelingprep.com/stable/cash-flow-statement"
}


def get_financial_statement(url, symbol, period="annual", limit=1):
    """Fetch single symbol financial statement (latest record only)."""
    api = f"{url}?symbol={symbol}&period={period}&limit={limit}&apikey={API_KEY}"
    r = requests.get(api)
    if r.status_code != 200:
        return None
    data = r.json()
    if not data:
        return None
    return data[0]  # only latest record


def build_financial_df(symbols, statement_name, url):
    rows = []
    for sym in tqdm(symbols):
        # print(f"Fetching {statement_name}: {sym}")
        data = get_financial_statement(url, sym)
        if data is None:
            continue
        rows.append(data)

    df = pd.DataFrame(rows).set_index("symbol")
    return df


def save_parquet(index_name, df, statement_name):
    filename = f"{index_name}_{statement_name}.parquet"
    df.to_parquet(OUTPUT_DIR / filename)
    print(f"Saved: {filename} shape={df.shape}")


def process_index_financials(index_name, constituent_file):
    df_const = pd.read_parquet(constituent_file)

    print(f"\n============================")
    print(f"Processing index: {index_name}, shape {df_const.shape}")
    print(f"============================")

    symbols = df_const.index.tolist()

    for statement_name, url in FIN_STAT_ENDPOINTS.items():
        print(f"\n--- {statement_name} for {index_name} ---")
        df_fin = build_financial_df(symbols, statement_name, url)
        save_parquet(index_name, df_fin, statement_name)


def main():
    constituent_files = list(INPUT_DIR.glob("*_constituents.parquet"))
    if not constituent_files:
        print("No constituents files found.")
        return
    for file in constituent_files:
        name_part = file.stem.replace("_constituents", "")
        index_name = "_".join(name_part.split("_")[:2])

        process_index_financials(index_name, file)


if __name__ == "__main__":
    main()


In [ ]:
pd.read_parquet("../database/financial_statements/NASDAQ_251128_Balance_Sheet.parquet")

In [2]:
import robin_stocks.robinhood as rh
login = rh.login(username="sl5661@columbia.edu",password="Lsq181313890")
profile = rh.profiles.load_account_profile()
profile

Starting login process...
ERROR: There was an issue loading pickle file. Authentication may be expired - logging in normally.
Verification required, handling challenge...
Starting verification process...
Check robinhood app for device approvals method...
Verification successful!


{'url': 'https://api.robinhood.com/accounts/985769173/',
 'portfolio_cash': '4899.0900',
 'can_downgrade_to_cash': 'https://api.robinhood.com/accounts/985769173/can_downgrade_to_cash/',
 'user': 'api.robinhood.com/user/',
 'account_number': '985769173',
 'type': 'margin',
 'brokerage_account_type': 'individual',
 'created_at': '2025-04-09T14:37:13.553292Z',
 'updated_at': '2026-01-06T03:30:17.987221Z',
 'deactivated': False,
 'deposit_halted': False,
 'withdrawal_halted': False,
 'only_position_closing_trades': False,
 'buying_power': '16038.0315',
 'onbp': '16038.0315',
 'cash_available_for_withdrawal': '4899.0900',
 'cash_available_for_withdrawal_without_margin': '4899.0900',
 'cash': '4899.0900',
 'amount_eligible_for_deposit_cancellation': '4899.0900',
 'cash_held_for_orders': '0.0000',
 'uncleared_deposits': '0.0000',
 'sma': '0',
 'sma_held_for_orders': '0',
 'unsettled_funds': '0.0000',
 'unsettled_debit': '0.0000',
 'crypto_buying_power': '4899.0900',
 'max_ach_early_access_amo

In [3]:
positions = rh.account.get_open_stock_positions()
positions

[{'url': 'https://api.robinhood.com/positions/985769173/b2e06903-5c44-46a4-bd42-2a696f9d68e1/',
  'instrument': 'https://api.robinhood.com/instruments/b2e06903-5c44-46a4-bd42-2a696f9d68e1/',
  'instrument_id': 'b2e06903-5c44-46a4-bd42-2a696f9d68e1',
  'symbol': 'BABA',
  'account': 'https://api.robinhood.com/accounts/985769173/',
  'account_number': '985769173',
  'brokerage_account_type': 'individual',
  'average_buy_price': '174.2888',
  'pending_average_buy_price': '174.2888',
  'quantity': '13.00000000',
  'intraday_average_buy_price': '0.0000',
  'intraday_quantity': '0.00000000',
  'shares_available_for_exercise': '13.00000000',
  'shares_available_for_sells': '13.00000000',
  'shares_held_for_buys': '0.00000000',
  'shares_held_for_sells': '0.00000000',
  'shares_held_for_stock_grants': '0.00000000',
  'shares_held_for_options_collateral': '0.00000000',
  'shares_held_for_options_events': '0.00000000',
  'shares_held_for_asset_transfer': '0.00000000',
  'shares_pending_from_opti

In [6]:
rh.stocks.get_latest_price("BABA")

['152.110000']

In [ ]:
bars = rh.stocks.get_stock_historicals(
    "AAPL",
    interval="day",
    span="month"
)

df = pd.DataFrame(bars)
df["close_price"] = df["close_price"].astype(float)
df

,begins_at,open_price,close_price,high_price,low_price,volume,session,interpolated,symbol
0,2025-11-28T00:00:00Z,277.260000,278.85,279.000000,275.986500,20135620,reg,False,AAPL
1,2025-12-01T00:00:00Z,278.010000,283.10,283.420000,276.140000,46587722,reg,False,AAPL
2,2025-12-02T00:00:00Z,283.000000,286.19,287.400000,282.630100,53669532,reg,False,AAPL
3,2025-12-03T00:00:00Z,286.200000,284.15,288.620000,283.300000,43538687,reg,False,AAPL
4,2025-12-04T00:00:00Z,284.095000,280.70,284.730000,278.590000,43989056,reg,False,AAPL
5,2025-12-05T00:00:00Z,280.540000,278.78,281.140000,278.050000,47265845,reg,False,AAPL
6,2025-12-08T00:00:00Z,278.130000,277.89,279.669300,276.150000,38211832,reg,False,AAPL
7,2025-12-09T00:00:00Z,278.160000,277.18,280.030000,276.920000,32193256,reg,False,AAPL
8,2025-12-10T00:00:00Z,277.750000,278.78,279.750000,276.440000,33038318,reg,False,AAPL
9,2025-12-11T00:00:00Z,279.095000,278.03,279.590000,273.810000,33247986,reg,False,AAPL


In [12]:
rh.orders.order_sell_market(
    symbol="BABA",
    quantity=1,
    timeInForce="gfd"
)

{'id': '69521778-62da-46b9-8fb1-0d9c06d2c056',
 'ref_id': '780765a9-de58-498a-84a4-59dbcf1f5e2d',
 'url': 'https://api.robinhood.com/orders/69521778-62da-46b9-8fb1-0d9c06d2c056/',
 'account': 'https://api.robinhood.com/accounts/985769173/',
 'user_uuid': 'be159967-b8ad-4464-a34a-a809d1184aa5',
 'position': 'https://api.robinhood.com/positions/985769173/b2e06903-5c44-46a4-bd42-2a696f9d68e1/',
 'cancel': 'https://api.robinhood.com/orders/69521778-62da-46b9-8fb1-0d9c06d2c056/cancel/',
 'instrument': 'https://api.robinhood.com/instruments/b2e06903-5c44-46a4-bd42-2a696f9d68e1/',
 'instrument_id': 'b2e06903-5c44-46a4-bd42-2a696f9d68e1',
 'cumulative_quantity': '0.00000000',
 'average_price': None,
 'fees': '0',
 'sec_fees': '0.00',
 'taf_fees': '0.00',
 'cat_fees': '0.00',
 'sales_taxes': [],
 'state': 'queued',
 'derived_state': 'queued',
 'pending_cancel_open_agent': None,
 'type': 'market',
 'side': 'sell',
 'time_in_force': 'gfd',
 'trigger': 'immediate',
 'price': None,
 'stop_price': N

In [9]:
"""
测试：通过 FMP stable API 获取某只股票某一天的 open / close / volume
Endpoint: https://financialmodelingprep.com/stable/historical-price-eod/full
"""

symbol = "AAPL"  # 想查询的股票列表
target_date = "2025-01-15"   # 想查询的日期

# ---- 方法 1: 用 stable API (推荐, 最新文档) ----
url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
params = {
    "symbol": symbol,
    "from": target_date,
    "to": target_date,
    "apikey": API_KEY,
}

resp = requests.get(url, params=params, timeout=15)
resp.raise_for_status()
data = resp.json()

if data:
    row = data[0]
    print(row)
    print(f"=== {symbol} on {target_date} (stable API) ===")
    print(f"  Open   : {row['open']}")
    print(f"  Close  : {row['close']}")
    print(f"  Volume : {row['volume']}")
    print(f"  High   : {row['high']}")
    print(f"  Low    : {row['low']}")
    print(f"  VWAP   : {row.get('vwap', 'N/A')}")
    print(f"  Change%: {row.get('changePercent', 'N/A')}")
else:
    print(f"No data returned for {symbol} on {target_date} (可能非交易日)")

# ---- 方法 2: 用 v3 API (alpha_utils.py 中使用的旧版) ----
url_v3 = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
params_v3 = {"from": target_date, "to": target_date, "apikey": API_KEY}
resp_v3 = requests.get(url_v3, params=params_v3, timeout=15)
data_v3 = resp_v3.json().get("historical", [])

if data_v3:
    row_v3 = data_v3[0]
    print(f"\n=== {symbol} on {target_date} (v3 API) ===")
    print(f"  Open   : {row_v3['open']}")
    print(f"  Close  : {row_v3['close']}")
    print(f"  Volume : {row_v3['volume']}")
else:
    print(f"\nv3 API: No data for {symbol} on {target_date}")

{'symbol': 'AAPL', 'date': '2025-01-15', 'open': 234.64, 'high': 238.96, 'low': 234.43, 'close': 237.87, 'volume': 39832000, 'change': 3.24, 'changePercent': 1.38, 'vwap': 236.475}
=== AAPL on 2025-01-15 (stable API) ===
  Open   : 234.64
  Close  : 237.87
  Volume : 39832000
  High   : 238.96
  Low    : 234.43
  VWAP   : 236.475
  Change%: 1.38

v3 API: No data for AAPL on 2025-01-15


In [15]:
url = "https://financialmodelingprep.com/stable/historical-price-eod/full?symbol=A&from=2020-01-01&to=2025-12-31&apikey=WsWl7TJA9XNRGsCZDiT0dLC5pVaYAPjn"
requests.get(url).json()

[{'symbol': 'A',
  'date': '2025-12-31',
  'open': 137.61,
  'high': 137.97,
  'low': 136.04,
  'close': 136.07,
  'volume': 950000,
  'change': -1.54,
  'changePercent': -1.12,
  'vwap': 136.9225},
 {'symbol': 'A',
  'date': '2025-12-30',
  'open': 137.57,
  'high': 138.16,
  'low': 136.83,
  'close': 137.62,
  'volume': 2423111,
  'change': 0.05,
  'changePercent': 0.03634513,
  'vwap': 137.545},
 {'symbol': 'A',
  'date': '2025-12-29',
  'open': 138.4,
  'high': 139.07,
  'low': 137.62,
  'close': 137.93,
  'volume': 1640851,
  'change': -0.47,
  'changePercent': -0.3396,
  'vwap': 138.255},
 {'symbol': 'A',
  'date': '2025-12-26',
  'open': 138.5,
  'high': 138.63,
  'low': 137.87,
  'close': 138.39,
  'volume': 739337,
  'change': -0.11,
  'changePercent': -0.07942238,
  'vwap': 138.3475},
 {'symbol': 'A',
  'date': '2025-12-24',
  'open': 138.35,
  'high': 138.57,
  'low': 137.77,
  'close': 138.32,
  'volume': 509000,
  'change': -0.03,
  'changePercent': -0.02168413,
  'vwap': 

In [26]:
# url = "https://financialmodelingprep.com/stable/historical-market-capitalization"
url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
params = {
    "symbol": "^IRX",
    "from": "2020-01-01",
    "to": "2025-12-31",
    "apikey": API_KEY,
}
resp = requests.get(url, params=params, timeout=15)
data = resp.json()
df = pd.DataFrame(data)
df
# 每条记录包含 date 和 marketCap 字段

,symbol,date,open,high,low,close,volume,change,changePercent,vwap
0,^IRX,2025-12-31,3.540,3.550,3.530,3.550,0,0.010,0.28249,3.54
1,^IRX,2025-12-30,3.560,3.560,3.540,3.540,0,-0.020,-0.56180,3.55
2,^IRX,2025-12-29,3.540,3.540,3.530,3.540,0,0.000,0.00000,3.54
3,^IRX,2025-12-26,3.550,3.550,3.540,3.540,0,-0.010,-0.28169,3.54
4,^IRX,2025-12-24,3.550,3.560,3.530,3.560,0,0.010,0.28169,3.55
...,...,...,...,...,...,...,...,...,...,...
1503,^IRX,2020-01-08,1.493,1.493,1.485,1.493,0,0.000,0.00000,1.49
1504,^IRX,2020-01-07,1.505,1.505,1.500,1.500,0,-0.005,-0.33223,1.50
1505,^IRX,2020-01-06,1.478,1.490,1.475,1.488,0,0.010,0.67659,1.48
1506,^IRX,2020-01-03,1.490,1.490,1.460,1.473,0,-0.017,-1.14000,1.47


In [ ]:
url = "https://financialmodelingprep.com/stable/historical-price-eod/full?symbol=AAPL&apikey=WsWl7TJA9XNRGsCZDiT0dLC5pVaYAPjn"
requests.get(url).json()

[{'symbol': 'AAPL',
  'date': '2026-02-13',
  'open': 262.01,
  'high': 262.23,
  'low': 255.45,
  'close': 255.78,
  'volume': 56290673,
  'change': -6.23,
  'changePercent': -2.38,
  'vwap': 258.8675},
 {'symbol': 'AAPL',
  'date': '2026-02-12',
  'open': 275.59,
  'high': 275.72,
  'low': 260.18,
  'close': 261.73,
  'volume': 81077229,
  'change': -13.86,
  'changePercent': -5.03,
  'vwap': 268.305},
 {'symbol': 'AAPL',
  'date': '2026-02-11',
  'open': 274.7,
  'high': 280.18,
  'low': 274.45,
  'close': 275.5,
  'volume': 51931300,
  'change': 0.805,
  'changePercent': 0.29123,
  'vwap': 276.2075},
 {'symbol': 'AAPL',
  'date': '2026-02-10',
  'open': 274.89,
  'high': 275.37,
  'low': 272.94,
  'close': 273.68,
  'volume': 34376900,
  'change': -1.2,
  'changePercent': -0.44018,
  'vwap': 274.22},
 {'symbol': 'AAPL',
  'date': '2026-02-09',
  'open': 277.9,
  'high': 278.2,
  'low': 271.7,
  'close': 274.62,
  'volume': 44623400,
  'change': -3.28,
  'changePercent': -1.18,
  'v